# 01 · Kiểm định Fashion-MNIST

**Mục đích:** xác minh bốn archive chính thức và hai split IDX trước khi dựng giao dịch.

**Điều kiện chạy:** Julia 1.13, môi trường `environments/fashion_notebooks`, internet ở lần tải đầu hoặc archive hợp lệ trong `data/raw/fashion-mnist/`.

**Đầu vào:** bốn file Fashion-MNIST `.gz` chính thức.

**Đầu ra:** `results/fashion/release/<run_id>/dataset/dataset_manifest.toml`, ảnh mẫu của đủ mười lớp và quy tắc ID ổn định `train:<index>` / `test:<index>` với index bắt đầu từ 0.


## 1. Tìm repository và tạo thư mục cho lượt chạy

Cell này tìm repo từ thư mục làm việc của kernel, kích hoạt môi trường Julia notebook và tạo một `run_id` mới. Kiểm tra đường dẫn repo, Julia 1.13 và run root dưới `results/fashion/release/`.


In [1]:
using Pkg

function find_repo_root(start=pwd())
    current = realpath(start)
    while true
        if isfile(joinpath(current, "Project.toml")) && isfile(joinpath(current, "src", "EclatFIM.jl"))
            return current
        end
        parent = dirname(current)
        parent == current && error("Không tìm thấy repo; mở notebook từ repo hoặc đặt FASHION_MNIST_REPO_ROOT")
        current = parent
    end
end

repo_root = get(ENV, "FASHION_MNIST_REPO_ROOT", find_repo_root())
notebook_env = joinpath(repo_root, "environments", "fashion_notebooks")
Pkg.activate(notebook_env)
using EclatFIM
using SHA: sha256
using TOML

@assert VERSION >= v"1.13"
data_dir = joinpath(repo_root, "data", "raw", "fashion-mnist")
run_id = EclatFIM.new_release_run_id()
run_root = joinpath(repo_root, "results", "fashion", "release", run_id)
mkpath(run_root)
println("Julia ", VERSION, " | repo: ", repo_root)
println("Raw data: ", data_dir)
println("Run ID: ", run_id)
println("Run root: ", relpath(run_root, repo_root))


  Activating project at `/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/environments/fashion_notebooks`
[ Info: Precompiling EclatFIM [f035e606-8504-4775-8232-c73e28f25aa6] (caches not reused: 1 for file size changed, 1 for different source file path)
Precompiling packages...


Julia 1

   2.2 s  ✓ EclatFIM
  1 dependency successfully precompiled in 2 seconds. 46 already precompiled.


.13.3 | repo: /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining
Raw data: /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/data/raw/fashion-mnist
Run ID: fashion_47265722452561
Run root: results/fashion/release/fashion_47265722452561


## 2. Tải hoặc mở lại bốn archive và kiểm tra SHA-256

Cell này gọi downloader đã ghim checksum chính thức; archive có sẵn chỉ được dùng sau khi hash khớp. Kiểm tra bốn dòng `SHA-256 OK` trước khi tiếp tục.


In [2]:
archive_paths = EclatFIM.download_fashion_mnist(data_dir)
archive_hashes = Dict{String,String}()
for (path, expected) in zip(archive_paths, EclatFIM.FASHION_FILES)
    actual = bytes2hex(open(sha256, path))
    @assert actual == expected.sha256
    archive_hashes[expected.name] = actual
    println("SHA-256 OK  ", expected.name, "  (", filesize(path), " bytes)")
end
@assert length(archive_hashes) == 4


SHA-256 OK  train-images-idx3-ubyte.gz  (26421880 bytes)
SHA-256 OK  train-labels-idx1-ubyte.gz  (29515 bytes)
SHA-256 OK  t10k-images-idx3-ubyte.gz  (4422102 bytes)
SHA-256 OK  t10k-labels-idx1-ubyte.gz  (5148 bytes)


## 3. Đọc ảnh và nhãn theo split gốc

Cell này đọc train và test IDX bằng parser của package. Kiểm tra shape `(28, 28, N)`, pixel `UInt8` và số nhãn của mỗi split.


In [3]:
train = EclatFIM.load_fashion_mnist(data_dir; split=:train)
test = EclatFIM.load_fashion_mnist(data_dir; split=:test)
println("Train: ", size(train.images), " | labels: ", length(train.labels))
println("Test:  ", size(test.images), " | labels: ", length(test.labels))
println("Pixel type/range: ", eltype(train.images), " / ", extrema(train.images))


Train: (28, 28, 60000) | labels: 60000
Test:  (28, 28, 10000) | labels: 10000
Pixel type/range: UInt8 / (0x00, 0xff)


## 4. Kiểm tra dimensions, lớp và split counts

Cell này xác nhận đủ 60.000 ảnh train, 10.000 ảnh test, ảnh 28×28, nhãn 0–9 và 6.000/1.000 ảnh mỗi lớp. Đọc bảng đếm lớp để phát hiện dữ liệu sai trước khi manifest được ghi.


In [4]:
@assert size(train.images) == (28, 28, 60_000)
@assert size(test.images) == (28, 28, 10_000)
@assert length(train.labels) == 60_000 && length(test.labels) == 10_000
@assert all(label -> 0 <= label <= 9, train.labels)
@assert all(label -> 0 <= label <= 9, test.labels)
train_counts = [count(==(label), train.labels) for label in 0:9]
test_counts = [count(==(label), test.labels) for label in 0:9]
@assert all(==(6_000), train_counts) && all(==(1_000), test_counts)
for label in 0:9
    println(lpad(string(label), 2), "  ", rpad(EclatFIM.FASHION_CLASSES[label + 1], 14),
        " train=", train_counts[label + 1], " test=", test_counts[label + 1])
end
println("PASS: 70,000 ảnh 28×28, đủ mười lớp.")


 0  T-shirt/top    train=6000 test=1000
 1  Trouser        train=6000 test=1000
 2  Pullover       train=6000 test=1000
 3  Dress          train=6000 test=1000
 4  Coat           train=6000 test=1000
 5  Sandal         train=6000 test=1000
 6  Shirt          train=6000 test=1000
 7  Sneaker        train=6000 test=1000
 8  Bag            train=6000 test=1000
 9  Ankle boot     train=6000 test=1000
PASS: 70,000 ảnh 28×28, đủ mười lớp.


## 5. Xem một ảnh mẫu cho mỗi lớp

Cell này dựng gallery PNG nhỏ ngay trong output của notebook, một ảnh train cho mỗi nhãn. Kiểm tra đúng mười lớp và ảnh hiển thị theo grayscale 28×28.


In [5]:
using Base64
using ColorTypes: Gray
using FixedPointNumbers: N0f8
using Markdown
using PNGFiles

function png_data_uri(image)
    path, io = mktemp()
    close(io)
    try
        PNGFiles.save(path, Gray.(reinterpret(N0f8, image)))
        return "data:image/png;base64," * base64encode(read(path))
    finally
        rm(path; force=true)
    end
end

cards = String[]
for label in 0:9
    index = findfirst(==(label), train.labels)
    image_uri = png_data_uri(@view train.images[:, :, index])
    push!(cards, "<figure style='margin:8px;text-align:center'><img width='112' height='112' style='image-rendering:pixelated' src='$image_uri'><figcaption>$(label): $(EclatFIM.FASHION_CLASSES[label + 1])</figcaption></figure>")
end
display(Markdown.HTML("<div style='display:flex;flex-wrap:wrap'>" * join(cards) * "</div>"))


[ Info: Precompiling PNGFiles [f57f5aa1-a3ce-4bc8-8ab9-96f992907883] (caches not reused: 1 for different dependency version already loaded)
Precompiling packages...
  17.1 s  ✓ ImageCore
   4.3 s  ✓ PNGFiles
  2 dependencies successfully precompiled in 21 seconds. 26 already precompiled.


HTML{String}("<div style='display:flex;flex-wrap:wrap'><figure style='margin:8px;text-align:center'><img width='112' height='112' style='image-rendering:pixelated' src=''><figcaption>0: T-shirt/top</figcaption></figure><figure style='margin:8px;text-align:center'><img width='112' height='112' style='image-rendering:pixelated' src=''><figcaption>1: Trouser</figcaption></figure><figure style='margin:8px;text-align:center'><img width='112' height='112' style='image-rendering:pixelated' src=''><figcaption>2: Pullover</figcaption></figure><figure style='margin:8px;text-align:center'><img width='112' height='112' style='image-rendering:pixelated' src='

## 6. Ghi manifest kiểm định cho các notebook sau

Cell này ghi run ID, archive paths và hashes, split counts, image shape, class count và quy tắc stable image ID. Nó đọc lại manifest và xác minh lại bốn archive; đầu ra cần kiểm tra là trạng thái `PASS` và đường dẫn manifest.


In [6]:
archive_relpaths = Dict(file.name => relpath(joinpath(data_dir, file.name), repo_root)
    for file in EclatFIM.FASHION_FILES)
manifest = EclatFIM.build_dataset_manifest(archive_relpaths, archive_hashes;
    run_id, train_count=length(train.labels), test_count=length(test.labels),
    run_root=relpath(run_root, repo_root))
manifest_path = joinpath(run_root, "dataset", "dataset_manifest.toml")
EclatFIM.write_dataset_manifest(manifest_path, manifest)
verified_manifest = EclatFIM.read_dataset_manifest(manifest_path;
    verify_archives=true, repo_root)
@assert verified_manifest["validation_status"] == "PASS"
@assert verified_manifest["image_id_examples"] == ["train:0", "train:59999", "test:0", "test:9999"]
println("Manifest: ", relpath(manifest_path, repo_root))
println("Stable IDs: train:0 … train:59999; test:0 … test:9999")


Manifest: results/fashion/release/fashion_47265722452561/dataset/dataset_manifest.toml
Stable IDs: train:0 … train:59999; test:0 … test:9999
